In [ ]:
from IPython.display import Image as IPImage
from IPython.display import Image, display

from typing import TypedDict, Optional, List, Dict, Any, Annotated, Tuple, Optional, Literal, Callable
from typing_extensions import TypedDict
import operator

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState
from langgraph.graph import StateGraph, START, END

from langgraph.prebuilt import tools_condition, ToolNode
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, BaseMessage
from langchain_openai import ChatOpenAI

from langgraph.graph.message import add_messages

### NEW
from datetime import datetime, timedelta, date
import pandas as pd
from langchain.tools import tool
from langchain_core.tools import StructuredTool
from langchain.agents import create_agent
from datetime import date, datetime, timedelta
import pandas as pd
from __future__ import annotations
from langchain_core.tools import tool
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
import csv
import re
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
import json
from langchain_core.documents import Document

load_dotenv(find_dotenv())

LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")
LANGCHAIN_PROJECT = os.getenv("LANGCHAIN_PROJECT")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
LANGCHAIN_TRACING_V2 = os.getenv("LANGCHAIN_TRACING_V2") == "true"

KB_DIR = Path.cwd() / "kb"

CARDIOLOGY_SCHEDULE_CSV = KB_DIR / "cardiology_appointment_slots.csv"

CARDIOLOGY_KB_PATH = "./kb/cardiology_kb.jsonl"

REBUILD_CHROMA = False   # <-- set to False to reuse persisted DB
CHROMA_DIR = "./chroma_kb"
#CHROMA_DIR = "./chroma_kb_rebuild" if REBUILD_CHROMA else "./chroma_kb"

print("LANGCHAIN_PROJECT:", LANGCHAIN_PROJECT)
print("LANGCHAIN_TRACING_V2:", LANGCHAIN_TRACING_V2)

In [ ]:
"""
Workout-History Trends Agent (LangChain + LangGraph)
Structured Planner → Tool Executor → Interpreter

Prereqs (typical):
  pip install -U langgraph langchain langchain-core langchain-openai pandas pydantic

You also need a SQLite DB with a `workouts` table.
If you used the synthetic generator earlier, point DB_PATH to your .sqlite file.

Example user prompt:
  "For MB-002, analyze my last 8 weeks: am I improving, and what’s driving intensity?"
"""

from __future__ import annotations

import datetime as dt
import re
import sqlite3
from typing import Any, Dict, List, Optional, TypedDict, Literal, Union

import pandas as pd
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END


# ----------------------------
# Config
# ----------------------------
# Point this to wherever your DB lives
DB_PATH = r"peloton_workouts.sqlite"  # e.g. r"C:\...\peloton_workouts.sqlite"

# Set this to your project "today"
TODAY = dt.date(2026, 3, 3)


# ----------------------------
# Helper: default timeframe
# ----------------------------
def default_last_8_weeks(today: dt.date) -> tuple[str, str]:
    # last 8 weeks ending yesterday
    end_date = today - dt.timedelta(days=1)
    start_date = end_date - dt.timedelta(weeks=8) + dt.timedelta(days=1)
    return start_date.isoformat(), end_date.isoformat()


# ----------------------------
# Tools
# ----------------------------
@tool
def read_workouts(
    member_id: str,
    start_date: str,
    end_date: str,
    types: Optional[List[str]] = None
) -> Dict[str, Any]:
    """
    Load workouts for a member between start_date and end_date (inclusive).
    Returns rows as JSON-serializable dicts.
    """
    q = """
    SELECT *
    FROM workouts
    WHERE member_id = ?
      AND date >= ?
      AND date <= ?
    """
    params: List[Any] = [member_id, start_date, end_date]

    if types:
        placeholders = ",".join(["?"] * len(types))
        q += f" AND type IN ({placeholders})"
        params.extend(types)

    with sqlite3.connect(DB_PATH) as conn:
        df = pd.read_sql_query(q, conn, params=params)

    return {"ok": True, "row_count": int(len(df)), "rows": df.to_dict(orient="records")}


@tool
def summarize_time_series(
    rows: List[Dict[str, Any]],
    metrics: List[str],
    freq: str = "W"
) -> Dict[str, Any]:
    """
    Aggregate metrics over time and compute simple trend direction.

    freq:
      'W' weekly, 'D' daily, 'M' monthly

    Returns:
      - by_period_mean: list of records {period, metric1, metric2, ...}
      - trends: per-metric slope_per_period (simple linear fit)
    """
    df = pd.DataFrame(rows)
    if df.empty:
        return {"ok": True, "note": "No workouts in timeframe.", "by_period_mean": [], "trends": {}}

    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date")

    # Period (start timestamp for each period)
    df["period"] = df["date"].dt.to_period(freq).dt.start_time

    # Only keep requested metrics that exist
    metrics_present = [m for m in metrics if m in df.columns]
    if not metrics_present:
        return {"ok": True, "note": "None of the requested metrics exist in the data.", "by_period_mean": [], "trends": {}}

    agg = df.groupby("period")[metrics_present].mean(numeric_only=True)

    # Trend slope via least squares on index
    trends: Dict[str, Any] = {}
    for m in metrics_present:
        series = agg[m].dropna()
        if len(series) < 3:
            trends[m] = {"slope_per_period": None, "note": "Not enough points for trend."}
            continue
        x = pd.Series(range(len(series)), dtype=float)
        y = series.reset_index(drop=True).astype(float)
        # slope = cov(x,y)/var(x)
        slope = float(((x - x.mean()) * (y - y.mean())).sum() / ((x - x.mean()) ** 2).sum())
        trends[m] = {"slope_per_period": slope}

    out = agg.reset_index()
    out["period"] = out["period"].dt.strftime("%Y-%m-%d")

    return {
        "ok": True,
        "metrics_used": metrics_present,
        "by_period_mean": out.to_dict(orient="records"),
        "trends": trends,
        "period_count": int(len(agg)),
    }


@tool
def zone_distribution(rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Compute percent time spent in HR zones overall and by type.
    Normalizes by sum(zone minutes) if available; otherwise uses duration_min.
    """
    df = pd.DataFrame(rows)
    if df.empty:
        return {"ok": True, "note": "No workouts.", "overall_zone_pct": {}, "by_type_zone_pct": {}}

    zone_cols = ["zone1_minutes", "zone2_minutes", "zone3_minutes", "zone4_minutes", "zone5_minutes"]
    for c in zone_cols:
        if c not in df.columns:
            df[c] = None

    # zone_total = sum of zones per workout (may be NaN if HR data missing)
    df["zone_total"] = df[zone_cols].sum(axis=1, numeric_only=True)
    denom = df["zone_total"].where(df["zone_total"] > 0, pd.to_numeric(df.get("duration_min"), errors="coerce"))

    def pct_for(sub: pd.DataFrame) -> Dict[str, Any]:
        zsum = sub[zone_cols].sum(numeric_only=True)
        d = float(pd.to_numeric(denom.loc[sub.index], errors="coerce").sum())
        if d <= 0:
            return {"note": "Missing zone minutes and duration for normalization."}
        return {z: float(zsum.get(z, 0.0) / d) for z in zone_cols}

    overall = pct_for(df)
    by_type: Dict[str, Any] = {}
    if "type" in df.columns:
        for t in sorted(df["type"].dropna().unique()):
            by_type[t] = pct_for(df[df["type"] == t])

    return {"ok": True, "overall_zone_pct": overall, "by_type_zone_pct": by_type}


@tool
def segment_by(
    rows: List[Dict[str, Any]],
    by: List[str],
    metrics: List[str]
) -> Dict[str, Any]:
    """
    Group workouts by dimensions and summarize metrics (mean).
    Supported derived fields:
      - weekday (from date)
      - time_bucket (from start_time_local HH:MM)
    """
    df = pd.DataFrame(rows)
    if df.empty:
        return {"ok": True, "note": "No workouts.", "segments": []}

    df["date"] = pd.to_datetime(df["date"])

    if "weekday" in by:
        df["weekday"] = df["date"].dt.day_name()

    if "time_bucket" in by:
        def bucket(t):
            if not isinstance(t, str) or not re.match(r"^\d{2}:\d{2}$", t):
                return "unknown"
            hh = int(t.split(":")[0])
            if 5 <= hh < 11:
                return "morning"
            if 11 <= hh < 17:
                return "midday"
            if 17 <= hh < 22:
                return "evening"
            return "late_night"
        df["time_bucket"] = df.get("start_time_local", "").apply(bucket)

    # Keep only metrics that exist
    metrics_present = [m for m in metrics if m in df.columns]
    if not metrics_present:
        return {"ok": True, "note": "None of the requested metrics exist in the data.", "segments": []}

    grouped = df.groupby(by)[metrics_present].mean(numeric_only=True).reset_index()
    return {"ok": True, "metrics_used": metrics_present, "segments": grouped.to_dict(orient="records")}


@tool
def detect_anomalies(rows: List[Dict[str, Any]], metric: str) -> Dict[str, Any]:
    """
    Detect outliers using IQR.
    Returns flagged workouts with workout_id/date/type/metric.
    """
    df = pd.DataFrame(rows)
    if df.empty or metric not in df.columns:
        return {"ok": True, "note": "No data or metric missing.", "outliers": []}

    s = pd.to_numeric(df[metric], errors="coerce").dropna()
    if len(s) < 8:
        return {"ok": True, "note": "Not enough points for anomaly detection.", "outliers": []}

    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr

    vals = pd.to_numeric(df[metric], errors="coerce")
    out = df[(vals < lo) | (vals > hi)]

    cols = [c for c in ["workout_id", "date", "type", metric] if c in out.columns]
    return {"ok": True, "bounds": {"low": float(lo), "high": float(hi)}, "outliers": out[cols].to_dict(orient="records")}


# ----------------------------
# Structured Plan Schema
# ----------------------------

class SummarizeArgs(BaseModel):
    metrics: List[str]
    freq: str = "W"

class SegmentArgs(BaseModel):
    by: List[str]
    metrics: List[str]

class AnomalyArgs(BaseModel):
    metric: str

class ZoneArgs(BaseModel):
    # no args needed
    pass

class SummarizeStep(BaseModel):
    tool: Literal["summarize_time_series"]
    args: SummarizeArgs

class ZoneStep(BaseModel):
    tool: Literal["zone_distribution"]
    args: ZoneArgs = Field(default_factory=ZoneArgs)

class SegmentStep(BaseModel):
    tool: Literal["segment_by"]
    args: SegmentArgs

class AnomalyStep(BaseModel):
    tool: Literal["detect_anomalies"]
    args: AnomalyArgs

PlanStep = Union[SummarizeStep, ZoneStep, SegmentStep, AnomalyStep]

class AnalysisPlan(BaseModel):
    member_id: str
    start_date: str
    end_date: str
    types: Optional[List[str]] = None
    steps: List[PlanStep] = Field(min_length=1, max_length=5)


# ----------------------------
# Prompts
# ----------------------------
PLAN_SYSTEM = """
You are a workout analytics planner.

Your job is to produce an ANALYSIS PLAN (not the final answer).

The plan must use the available analysis tools to answer the user's question.

AVAILABLE TOOLS:

1) summarize_time_series(rows, metrics, freq)
   - Use this when the user asks about improvement, trends, or change over time.
   - Typically use weekly frequency: freq="W".
   - Metrics must be explicitly provided.

2) zone_distribution(rows)
   - Use this when the user asks about intensity, heart rate zones, or effort distribution.

3) segment_by(rows, by, metrics)
   - Use this to determine what is driving performance.
   - Common segment dimensions:
       - "type"
       - "weekday"
       - "time_bucket"
   - Metrics must be explicitly provided.

4) detect_anomalies(rows, metric)
   - Use this to identify unusual spikes or drops in a metric.


IMPORTANT RULES:

- The plan must contain 1–5 steps.
- Each step must include the required arguments for that tool.
- If the user does not specify a date range, use the provided default start_date and end_date.
- Do NOT include the rows argument (it will be injected automatically).
- Do NOT write the final narrative response.
- Do NOT reference peers or infer individual peer data.
- This is a planning step only.


METRIC GUIDANCE:

Always safe to use:
  - "duration_min"
  - "calories"
  - "strive_score"

For bike or row workouts:
  - "output_kj"

For tread workouts:
  - "miles"
  - "average_speed_mph"

When unsure, default to:
  metrics = ["duration_min", "strive_score", "calories"]


COMMON PATTERNS:

If the user asks:
- "Am I improving?" → use summarize_time_series.
- "What is driving intensity?" → use zone_distribution and/or segment_by.
- "Any unusual drops or spikes?" → use detect_anomalies.
- "What factors influence performance?" → use segment_by.


The output must strictly conform to the AnalysisPlan schema.
Return only the structured plan object.
"""

INTERPRET_SYSTEM = """You are a supportive workout analytics coach.

You will be given:
- the user's request
- computed analysis results from tools

Explain insights in plain language, supportive and non-judgmental.
Guardrails:
- No medical/health advice or diagnoses.
- No peer comparisons or individual peer inference.
- If data is missing, say so and adapt.

Return this structure:
1) Key takeaways (3 bullets)
2) Trends (short paragraph)
3) What might be driving it (2–3 bullets)
4) Suggested experiments (2–3 bullets; general fitness guidance)
5) Data limitations (if any)
"""


# ----------------------------
# LangGraph State
# ----------------------------
class TrendState(TypedDict, total=False):
    messages: List[BaseMessage]
    plan: Dict[str, Any]
    tool_results: Dict[str, Any]
    response: str


# ----------------------------
# LLMs
# ----------------------------
base_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
planner_llm = base_llm.with_structured_output(AnalysisPlan, method="function_calling")  # <-- key: structured plan
interpreter_llm = base_llm  # free-form narrative is fine here


# ----------------------------
# Nodes
# ----------------------------
def make_plan_node(state: TrendState) -> TrendState:
    user_text = next((m.content for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), "")

    # Require member_id deterministically (it’s necessary to read workouts)
    m = re.search(r"\bMB-\d{3}\b", user_text, flags=re.I)
    if not m:
        ask = "What’s your member id (e.g., MB-001)? If you don’t specify dates, I’ll analyze the last 8 weeks."
        return {**state, "messages": state["messages"] + [AIMessage(content=ask)]}

    member_id = m.group(0).upper()
    sd, ed = default_last_8_weeks(TODAY)

    # Give defaults as context; the model must still return a valid AnalysisPlan object
    plan: AnalysisPlan = planner_llm.invoke([
        AIMessage(content=PLAN_SYSTEM),
        HumanMessage(content=f"User request: {user_text}\nDefaults if missing: member_id={member_id}, start_date={sd}, end_date={ed}")
    ])

    # Enforce required values
    plan.member_id = member_id
    if not plan.start_date or not plan.end_date:
        plan.start_date, plan.end_date = sd, ed

    # Cap steps defensively (still not “kludge”; it’s a policy)
    plan.steps = plan.steps[:5]

    return {**state, "plan": plan.model_dump()}


def run_tools_node(state: TrendState) -> TrendState:
    plan = state.get("plan")
    if not plan:
        return state

    # Always read workouts once
    base = read_workouts.invoke({
        "member_id": plan["member_id"],
        "start_date": plan["start_date"],
        "end_date": plan["end_date"],
        "types": plan.get("types"),
    })
    rows = base.get("rows", [])

    results: Dict[str, Any] = {
        "read_workouts": {
            "ok": base.get("ok", False),
            "row_count": base.get("row_count", 0),
            "member_id": plan["member_id"],
            "start_date": plan["start_date"],
            "end_date": plan["end_date"],
            "types": plan.get("types"),
        }
    }

    if not rows:
        # No need to run further tools
        state["tool_results"] = results
        return state

    # Execute planned steps
    for i, step in enumerate(plan.get("steps", []), start=1):
        tool_name = step["tool"]
        args = dict(step.get("args", {}))

        # Inject rows for analysis tools
        args.setdefault("rows", rows)

        if tool_name == "summarize_time_series":
            out = summarize_time_series.invoke(args)
        elif tool_name == "zone_distribution":
            out = zone_distribution.invoke(args)
        elif tool_name == "segment_by":
            out = segment_by.invoke(args)
        elif tool_name == "detect_anomalies":
            out = detect_anomalies.invoke(args)
        else:
            out = {"ok": False, "error": f"Unexpected tool: {tool_name}"}

        results[f"step_{i}_{tool_name}"] = out

    return {**state, "tool_results": results}


def interpret_node(state: TrendState) -> TrendState:
    user_text = next((m.content for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), "")
    results = state.get("tool_results", {})
    plan = state.get("plan", {})

    # Handle no member_id path
    if not plan:
        # likely asked a question for member_id
        return state

    # Handle “no workouts found”
    if results.get("read_workouts", {}).get("row_count", 0) == 0:
        msg = (
            f"I didn’t find any workouts for **{plan.get('member_id')}** between "
            f"**{plan.get('start_date')}** and **{plan.get('end_date')}**. "
            "If you want, try a different date range."
        )
        return {**state, "response": msg, "messages": state["messages"] + [AIMessage(content=msg)]}

    resp = interpreter_llm.invoke([
        AIMessage(content=INTERPRET_SYSTEM),
        HumanMessage(content=f"User request: {user_text}\n\nTool results:\n{results}")
    ]).content

    return {**state, "response": resp, "messages": state["messages"] + [AIMessage(content=resp)]}


# ----------------------------
# Build graph
# ----------------------------
builder = StateGraph(TrendState)
builder.add_node("plan", make_plan_node)
builder.add_node("run_tools", run_tools_node)
builder.add_node("interpret", interpret_node)

builder.set_entry_point("plan")

def route_after_plan(state: TrendState):
    # If plan missing (asked for member_id), stop
    return END if not state.get("plan") else "run_tools"

builder.add_conditional_edges("plan", route_after_plan, {END: END, "run_tools": "run_tools"})
builder.add_edge("run_tools", "interpret")
builder.add_edge("interpret", END)

trend_graph = builder.compile()

In [ ]:
display(Image(trend_graph.get_graph().draw_mermaid_png()))

<br><br><br>

<hr style="border:30px solid coral "> </hr>
<hr style="border:2px solid coral "> </hr>


# Demonstration

<hr style="border:2px solid coral "> </hr>


In [ ]:
# ----------------------------
# Example invocation
# ----------------------------
if __name__ == "__main__":
    result = trend_graph.invoke({
        "messages": [
            HumanMessage(content="For MB-002, analyze my last 8 weeks: am I improving, and what’s driving intensity?")
        ]
    })
    print(result.get("response", "(no response)"))